# CodeGen

Vamos a ver uno de los numerosos modelos base preentrenados para investigar su comportamiento en el *fine-tuning* y su inspección. Cabe acotar que estos modelos se utilizan únicamente con fines de investigación y son eficientes para nuestro propósito, por lo que no están pensados para puesta en producción ni despliegues reales.

Afortunadamente, la mayoría de estos modelos cuentan con versiones mucho más grandes y con una mayor cantidad de parámetros; por ello, en este contexto empleamos versiones más pequeñas, lo que nos facilita los cálculos, reduce los tiempos de importación y optimiza el flujo de trabajo.

Veremos un modelo llamado **CodeGen**, desarrollado y publicado por Salesforce, el cual está diseñado específicamente para la generación de código. Este modelo se ofrece en diferentes tamaños y versiones; en nuestro caso, utilizaremos la versión más pequeña, que cuenta con **350 millones de parámetros**.

No obstante, como se puede observar en su documentación oficial, también existen versiones masivas de hasta **16.000 millones de parámetros** (lo que representa dos órdenes de magnitud más de parámetros que la versión que emplearemos si se requieren respuestas de mayor calidad). Aun así, los modelos más compactos que utilizaremos cumplen perfectamente con nuestros propósitos de investigación.

> **Nota:** Podemos descargar este modelo desde Hugging Face, el cual podemos utilizar básicamente con la misma estructura que los modelos anteriores.

Puedes consultar el código fuente y la documentación técnica en el repositorio oficial de GitHub: [Salesforce/CodeGen](https://github.com/salesforce/CodeGen).

In [1]:
# Llamado Codegen-350M-mono que quiere decir que fue entrenado con un solo lenguaje de progeamación (Python)

In [2]:
# Tambien ofrecen una version multilenguaje que se ha formado con otros lenguajes de códificación

In [1]:
import numpy as np
import torch
import torch.nn as nn

# Configuración del dispositivo (GPU o CPU)
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM

# Cargar el tokenizador y el modelo CodeGen de 350M desde Hugging Face
tokenizer = AutoTokenizer.from_pretrained('Salesforce/codegen-350M-mono')
model = AutoModelForCausalLM.from_pretrained('Salesforce/codegen-350M-mono').to(device)

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  797MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/165 [00:00<?, ?it/s]

[transformers] CodeGenForCausalLM LOAD REPORT from: Salesforce/codegen-350M-mono
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...19}.attn.causal_mask | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors: reconstructing file:   0%|          |  0.00B /  797MB            

model.safetensors: downloading bytes:           |  0.00B            

In [4]:
model

CodeGenForCausalLM(
  (transformer): CodeGenModel(
    (wte): Embedding(51200, 1024)
    (drop): Dropout(p=0.0, inplace=False)
    (h): ModuleList(
      (0-19): 20 x CodeGenBlock(
        (ln_1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (attn): CodeGenAttention(
          (attn_dropout): Dropout(p=0.0, inplace=False)
          (resid_dropout): Dropout(p=0.0, inplace=False)
          (qkv_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (out_proj): Linear(in_features=1024, out_features=1024, bias=False)
        )
        (mlp): CodeGenMLP(
          (fc_in): Linear(in_features=1024, out_features=4096, bias=True)
          (fc_out): Linear(in_features=4096, out_features=1024, bias=True)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.0, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=1024, out_features=51200, bias=True)
)

¿Qué podemos observar? Al no tratarse de un modelo de OpenAI, algunos nombres de sus componentes pueden variar; sin embargo, ya conocemos los diferentes elementos que conforman un modelo de lenguaje.

Primero, contamos con un *Word Token Embedding* de $51.200$ y una dimensión de *embeddings* de $1024$. ¿Qué notamos después? Que la matriz de *Word Position Embedding* (WPE) no está presente en este modelo. A continuación, tenemos $20$ bloques *Transformer*, que comienzan con una capa de normalización como es de esperar.

Luego encontramos la matriz de proyección QKV comprimida en tres pesos de atención concatenados en una sola matriz (cabe destacar que estas matrices son cuadradas) de $1024$, las cuales se concatenan juntas para formar una matriz de $3072$ dimensiones. Aquí también podemos ver la matriz de proyección de salida $W_0$, que realiza la mezcla lineal de la matriz.

Posteriormente, tenemos el bloque MLP, que cuenta con una entrada y una salida, presentando la típica expansión por un factor de $4$ para después pasar a una contracción que devuelve el tensor a su dimensión original, acompañada de una función de activación no lineal GELU. Lo que no observamos aquí es la normalización de la capa al principio del MLP; no podemos explicar exactamente por qué tomaron esta decisión de diseño, pero es un claro ejemplo de cómo las diferentes empresas y grupos de investigación toman distintas elecciones arquitectónicas.

En conclusión: la estructura general de un LLM es la misma, pero siempre existirá una gran variedad de implementaciones y variaciones entre modelos.

## Visualización del Tokenizador

In [5]:
# Un poco sobre su tokenizador
print(f'El tokenizador tiene {tokenizer.vocab_size:,} tokens.\nAlgunos tokens aleatorios:\n')
for i in range(30):
  # generar un token aleatorio
  randtok = torch.randint(tokenizer.vocab_size, (1,))
  print(f'Token {randtok[0]:5} es "{tokenizer.decode(randtok)}"')

El tokenizador tiene 50,257 tokens.
Algunos tokens aleatorios:

Token  8179 es " busy"
Token 23787 es " shades"
Token 40613 es "Empty"
Token 49846 es "Mand"
Token 35115 es " fermentation"
Token 48625 es "Oracle"
Token   966 es " point"
Token 46778 es " sweeps"
Token 14825 es " funeral"
Token  4611 es "icy"
Token 14066 es " grateful"
Token 21025 es " punct"
Token 44281 es "clave"
Token 24332 es " Zimmerman"
Token 34069 es " depleted"
Token 23079 es "luence"
Token   660 es "te"
Token  5308 es "Me"
Token 50243 es " harsher"
Token 42801 es "Si"
Token 24356 es " 143"
Token 32018 es " Believe"
Token  4431 es " accur"
Token 33372 es "397"
Token  3047 es " training"
Token 36813 es "chlor"
Token  9322 es " employed"
Token  1968 es "ript"
Token 28596 es " summons"
Token 33378 es " torment"


El tokenizador tiene $50.257$ tokens; sin embargo, la matriz de *embeddings* es ligeramente mayor que el vocabulario. Esto significa que habrá varias filas en dicha matriz que no tienen conexión con los datos ni verán información real durante el entrenamiento.

Aunque estas filas se modificarán aleatoriamente durante el *backpropagation*, no corresponden a ningún token en específico. Se sospecha que este diseño se implementó únicamente para hacer que la matriz sea más eficiente en la GPU por razones de tiempo de cómputo y optimización de hardware.

In [6]:
text = "for i in range(10):"
input_ids = tokenizer(text, return_tensors="pt").input_ids.to(device)

generated_ids = model.generate(
    input_ids,
    max_length=222,
    temperature=1.4,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id,
)
print(tokenizer.decode(generated_ids[0], skip_special_tokens=True))

for i in range(10):
        r = i % 3
    answer.append(r) 
res = set(answer[0::3])    
print(list(res))    
   
# 주요 풀이

# n = int(input())
# answer = []
#
# a=[]
# for i in range(10):
#     a.append(int(input()))
    
#
# d=dict(Counter(a))
# answer = []
#
# key = sorted(list(d.keys()),reverse='True')
# dic=sorted(d.items(),key=itemgetter(1))
# ans=dic[0][1]
# if dic==dic:
#     if len(answer)==0:
#     	print(-1)
#     else:
#         for j in range(2):ans-(dic)
            


# Generar mas Código del modelo

In [10]:
text = 'for i in range(10):'
input_ids = tokenizer(text, return_tensors='pt').input_ids.to(device)

generated_ids = model.generate(
    input_ids,
    max_length=222,
    temperature=1.4,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id
)

print(tokenizer.decode(generated_ids[0], skip_special_tokens=True))

for i in range(10): print i
       output is 1..10
print ('*'.upper())       
output='*'     
list=list(range(0,20,2))              
print ('-'*10)                        
for i in range(len(list)): # iterate by length of a list         
   output+=list[i]+'*'              
   if i < len(list) - 2:       # If last char of list to 1
       output=output+'-' 
print (' '   * 20 +output  )       
del list[7:10]            #delect 10th element from list                       
output='-'   *20 + ' '       
print(' '.join(list)       
output=''.join(i for i in output if i.isdigit())  
print(' ' * 10+ output,' '.join(map(str, list)))     
print ('*'.upper())


 Este es el modelo preentrenado donde claramente parece código, pero no es necesariamente tan sensato.

In [13]:
# si usamos el codigo generado por el modelo mas grande probablemente tendremos mejores resultados y compilaria

In [12]:
for i in range(10): print i
       output is 1..10
print ('*'.upper())
output='*'
list=list(range(0,20,2))
print ('-'*10)
for i in range(len(list)): # iterate by length of a list
   output+=list[i]+'*'
   if i < len(list) - 2:       # If last char of list to 1
       output=output+'-'
print (' '   * 20 +output  )
del list[7:10]            #delect 10th element from list
output='-'   *20 + ' '
print(' '.join(list)
output=''.join(i for i in output if i.isdigit())
print(' ' * 10+ output,' '.join(map(str, list)))
print ('*'.upper())

SyntaxError: Missing parentheses in call to 'print'. Did you mean print(...)? (1528731355.py, line 1)

In [14]:
import os
import subprocess

# URL del repositorio de GitHub que deseas procesar (puedes cambiarla por cualquier otro repo con código Python)
repo_url = 'https://github.com/psf/requests.git'
repo_dir = 'requests_repo'
subprocess.run(['git', 'clone', repo_url, repo_dir])

# Inicializar una lista para todos los tokens
all_tokens = []

# Recorrer todos los archivos del repositorio buscando scripts de Python (.py)
for root, dirs, files in os.walk(repo_dir):
    for file in files:
        if file.endswith('.py'):

            # Cargar y procesar cada archivo de código
            file_path = os.path.join(root, file)
            try:
                with open(file_path, 'r', encoding='utf-8') as f:
                    code_content = f.read()

                # Tokenizar el código y agregarlo a la lista de tokens
                all_tokens += tokenizer(code_content)['input_ids']
            except Exception as e:
                # Omitir archivos que puedan dar errores de codificación
                continue

# Verificar el tamaño final del conjunto de datos procesado
print(f'\n\nEl conjunto de datos de entrenamiento contiene {len(all_tokens):,} tokens, de los cuales {len(set(all_tokens)):,} son únicos.')

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (4948 > 2048). Running this sequence through the model will result in indexing errors




El conjunto de datos de entrenamiento contiene 121,732 tokens, de los cuales 4,765 son únicos.


Es interesante pensar que el codigo que escribimos es redundante de la misma manera que tenemos una gran variedad de palabras y tokens cuando se trata de lenguaje natural

No todos los modelos son como ChatGPT. Numerosas personas, equipos de investigación, universidades, organizaciones sin fines de lucro y empresas con fines de lucro han desarrollado modelos lingüísticos diseñados a medida para fines específicos, o bien se han entrenado únicamente con conjuntos de datos concretos.